In [1]:
# --------------------------------------------------
# 0. Import
# --------------------------------------------------

import pandas as pd
import os

In [2]:
# --------------------------------------------------
# 1. Define input files
# --------------------------------------------------

file_info = [
    {
        "path": "ml_pair_level_predictions_with_probability.csv",
        "method": "Traditional Machine Learning"
    },
    {
        "path": "cnn_pair_level_predictions_with_probability.csv",
        "method": "Neural Network"
    },
    {
        "path": "bert_scibert_pair_level_predictions_with_probability.csv",
        "method": "BERT-based"
    },
    {
        "path": "chatgpt_zero_shot_predictions_with_probability.csv",
        "method": "LLM"
    },
    {
        "path": "chatgpt_few_shot_predictions_with_probability.csv",
        "method": "LLM"
    }
]

In [3]:
# --------------------------------------------------
# 2. Function to compute TP, FP, FN, TN
# --------------------------------------------------

def compute_error_summary(df, method_name):
    # Make sure labels are numeric
    df["TrueLabel"] = df["TrueLabel"].astype(int)
    df["PredictedLabel"] = df["PredictedLabel"].astype(int)
    
    rows = []
    
    for model_name, model_df in df.groupby("Model"):
        y_true = model_df["TrueLabel"]
        y_pred = model_df["PredictedLabel"]
        
        # Confusion matrix components
        tp = ((y_true == 1) & (y_pred == 1)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()
        tn = ((y_true == 0) & (y_pred == 0)).sum()
        
        # Error rates
        # FP_Rate = false positives among all actual NonMatch pairs
        # FN_Rate = false negatives among all actual Match pairs
        fp_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
        fn_rate = fn / (fn + tp) if (fn + tp) > 0 else 0
        
        rows.append({
            "Method": method_name,
            "Model": model_name,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "FP_Rate": fp_rate,
            "FN_Rate": fn_rate
        })
    
    return pd.DataFrame(rows)

In [4]:
# --------------------------------------------------
# 3. Read all files and combine error summaries
# --------------------------------------------------

summary_tables = []

for item in file_info:
    file_path = item["path"]
    method_name = item["method"]
    
    df = pd.read_csv(file_path, encoding="utf-8-sig")
    
    # Keep only rows with valid predictions
    df = df.dropna(subset=["TrueLabel", "PredictedLabel"]).copy()
    
    temp_summary = compute_error_summary(df, method_name)
    summary_tables.append(temp_summary)

error_summary_df = pd.concat(summary_tables, ignore_index=True)

In [5]:
# --------------------------------------------------
# 4. Optional: set model order and display names
# --------------------------------------------------

model_order = [
    "LogisticRegression",
    "RandomForest",
    "CNN",
    "BERT",
    "SciBERT",
    "ChatGPT_zero_shot",
    "ChatGPT_few_shot"
]

model_name_map = {
    "LogisticRegression": "LR",
    "RandomForest": "RF",
    "CNN": "CNN",
    "BERT": "BERT",
    "SciBERT": "SciBERT",
    "ChatGPT_zero_shot": "ChatGPT ZS",
    "ChatGPT_few_shot": "ChatGPT FS"
}

error_summary_df["Model"] = error_summary_df["Model"].replace(model_name_map)

display_order = [model_name_map[m] for m in model_order]

error_summary_df["Model"] = pd.Categorical(
    error_summary_df["Model"],
    categories=display_order,
    ordered=True
)

error_summary_df = error_summary_df.sort_values("Model").reset_index(drop=True)

In [6]:
# --------------------------------------------------
# 5. Format error rates
# --------------------------------------------------

error_summary_df["FP_Rate"] = error_summary_df["FP_Rate"].round(4)
error_summary_df["FN_Rate"] = error_summary_df["FN_Rate"].round(4)

In [7]:
# --------------------------------------------------
# 6. Save output
# --------------------------------------------------

error_summary_df.to_csv(
    "error_analysis_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print(error_summary_df)

                         Method       Model   TP   FP   FN    TN  FP_Rate  \
0  Traditional Machine Learning          LR  517  117   49  2713   0.0413   
1  Traditional Machine Learning          RF  424   15  142  2815   0.0053   
2                Neural Network         CNN  388  430  178  2400   0.1519   
3                    BERT-based        BERT  526   27   40  2803   0.0095   
4                    BERT-based     SciBERT  522   30   44  2800   0.0106   
5                           LLM  ChatGPT ZS  430    9  136  2821   0.0032   
6                           LLM  ChatGPT FS  449    7  117  2823   0.0025   

   FN_Rate  
0   0.0866  
1   0.2509  
2   0.3145  
3   0.0707  
4   0.0777  
5   0.2403  
6   0.2067  
